# CeNN-MemoryGraph V4 — semantic cellular memory for Qwen3.5-0.8B

**Research experiment · Qwen full-attention layer 11 · frozen pretrained Q/K/V, RoPE, QK norms and output projection.**

This notebook answers the question raised by CeNN-DeltaCell V3: *Does learned sparse communication among persistent memory cells improve Qwen layer fidelity after controlling for the benefits of exact Local32 attention?*

Predeclared controls: **V3.1 Delta+Local32**, **V3 Adaptive+Local32**, **V4 static learned graph+Local32**, **V4 content-adaptive graph+query router**, and **V4 shuffled topology+query router**. Two-stage calibration first fits a **shared** V3.1 core; every candidate then receives the same extra update budget from that state. A post-hoc frozen-weights edge-shuffle test isolates the trained graph topology.

**Important:** this is NOT a trained release, a GPU speedup claim, or a demonstrated novel result. It intentionally disallows cached generation and padded batches. Quick mode is screening; research mode is larger but still needs multiple random seeds for publication. Native Qwen is never overwritten.

## 0 · Setup
Select **Runtime → Change runtime type → NVIDIA GPU**. Clone the repository containing the real V4 module; unlike the CeNN-X2 notebook, this notebook **does not embed a second architecture copy**. Dependencies match the existing Qwen3.5 reference notebooks.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"

In [ ]:
import os,sys,subprocess, time, gc, json, hashlib, random, statistics, math, copy
from pathlib import Path
from contextlib import contextmanager
import numpy as np, torch, pandas as pd, matplotlib.pyplot as plt
from torch.nn import functional as F
REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3, Qwen35DeltaCellAttention
from tinycenn_lm.cenn_memorygraph_v4 import CeNNMemoryGraphV4
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Git SHA',GIT_SHA,'Torch',torch.__version__)
# Fail visibly if version-drift changed the integration interface.
assert torch.cuda.is_available(), 'The full Qwen3.5 experiment requires an NVIDIA GPU'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU:',torch.cuda.get_device_name(0),'dtype:',DTYPE)
subprocess.run(['nvidia-smi'],check=False)
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_cenn_memorygraph_v4.py')],check=True)

## 1 · Preset and ablation design
**quick:** 128-token windows, 80 shared baseline steps + 80 steps per candidate. **research:** 512-token windows, more data and optimization. Each candidate uses the **same sampling indices** for post-warmstart calibration, not merely the same number of updates. Both modes retain original Qwen weights and unpadded text input.

To isolate **graph** vs **local attention**, the static graph has the same fixed Local32 gate as V3.1, and the adaptive graph adds both content-dependent graph weights and per-token routing. A shuffled graph is a structural null but can retrain to compensate; the post-hoc frozen shuffle is stronger evidence of graph importance.

In [ ]:
# @title Experiment configuration
MODEL_ID='Qwen/Qwen3.5-0.8B'
PROFILE='quick' # @param ['quick','research']
TARGET_LAYERS=[11]  # Optional full attention targets: [3,7,11,15,19,23]
SEQ_LEN=128 if PROFILE=='quick' else 512
CALIB_BLOCKS=32 if PROFILE=='quick' else 128
DEV_BLOCKS=24 if PROFILE=='quick' else 48
TEST_BLOCKS=24 if PROFILE=='quick' else 64
SHARED_STEPS=80 if PROFILE=='quick' else 400
CANDIDATE_STEPS=80 if PROFILE=='quick' else 400
BATCH_SIZE=2
LR=2e-3  # Research default; V3 used 5e-3. Tune only using validation, not TEST.
FEATURE_DIM=32
CHUNK_SIZE=32
GRAPH_K=4
EDGE_DIM=12
SEED=2031
# Every candidate starts from same calibrated Delta+Local32 weights.
CANDIDATES={
 'delta_local32':dict(family='v3',mode='none',local_window=32),
 'v3_adaptive_local32':dict(family='v3',mode='adaptive',local_window=32),
 'graph_static_local32':dict(family='v4',graph_mode='learned',local_window=32,
                             content_adaptive=False,query_router=False),
 'graph_adaptive_local32':dict(family='v4',graph_mode='learned',local_window=32,
                               content_adaptive=True,query_router=True),
 'graph_shuffled_local32':dict(family='v4',graph_mode='shuffled',local_window=32,
                               content_adaptive=True,query_router=True),
}
OUT=Path('/content/cenn_memorygraph_v4_qwen35')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('profile',PROFILE,'layers',TARGET_LAYERS,'candidates',list(CANDIDATES))

## 2 · GPU numerical preflight
The repository test suite runs at setup. This extra GPU smoke test checks streaming parity, the graph gradient, and mixed-precision behavior with a non-divisible sequence length.

In [ ]:
torch.manual_seed(SEED)
unit=CeNNMemoryGraphV4(4,2,8,feature_dim=7,chunk_size=5,top_k=3,
                       local_window=5,graph_mode='learned',query_router=True).to(DEVICE)
q=torch.randn(1,4,23,8,device=DEVICE);k=torch.randn(1,2,23,8,device=DEVICE);v=torch.randn(1,2,23,8,device=DEVICE)
full,final_state=unit(q,k,v,return_state=True)
parts=[];state=None
for lo,hi in [(0,1),(1,6),(6,15),(15,23)]:
    y,state=unit(q[:,:,lo:hi],k[:,:,lo:hi],v[:,:,lo:hi],state,True)
    parts.append(y)
error=(full-torch.cat(parts,2)).abs().max().item()
assert error<3e-5,error
(full.square().mean()).backward()
assert unit.query_local_router.grad is not None and unit.edge_q.grad is not None
assert unit.edge_q.grad.abs().sum().item()>0
print('GPU streaming max diff',error,'state bytes',final_state.nbytes)
del unit,q,k,v,full,parts,state,final_state;torch.cuda.empty_cache()

## 3 · Pin teacher model and verify correct module interfaces
Qwen3.5-0.8B contains hybrid Gated DeltaNet layers and full-attention layers at indices **3, 7, 11, 15, 19, 23**. We replace only a single **full-attention token mixer** at a time. Pretrained norms, MLP, residuals, Q/K/V projections and output gate remain unchanged. This is a *text-only, cache-free* experiment.

In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
FULL=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert cfg.model_type=='qwen3_5_text',cfg.model_type
assert FULL==[3,7,11,15,19,23],FULL
assert all(i in FULL for i in TARGET_LAYERS), (FULL,TARGET_LAYERS)
print('Pinned Hugging Face model SHA:',REVISION,'layers:',cfg.num_hidden_layers,'full attention:',FULL)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(
    MODEL_ID,revision=REVISION,dtype=DTYPE,attn_implementation='sdpa',
    low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
for i in TARGET_LAYERS:
    native=LAYERS[i].self_attn
    print('layer',i,'q-heads',native.q_proj.out_features//(2*native.head_dim),
          'kv-heads',native.k_proj.out_features//native.head_dim,
          'head_dim',native.head_dim)
print('Total native parameters:',sum(p.numel() for p in model.parameters()))

## 4 · Independent train / validation / held-out test data
WikiText-2 raw train, validation, and test are disjoint. Each block is fixed length with no padding. The TEST split is not inspected until **after** choosing the winner on validation. Fixed metadata/fingerprints are saved to the final audit.

In [ ]:
from datasets import load_dataset

def windows_for_split(split, count, length):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    ids=[]
    for item in ds:
        sentence=item['text'].strip()
        if not sentence: continue
        ids.extend(tokenizer.encode(sentence,add_special_tokens=False))
        ids.append(tokenizer.eos_token_id)
        if len(ids)>=(count+1)*length: break
    if len(ids)<count*length:
        raise RuntimeError(f'Not enough text for {split}: {len(ids)} tokens')
    return torch.tensor(ids[:count*length],dtype=torch.long).reshape(count,length)
TRAIN=windows_for_split('train',CALIB_BLOCKS,SEQ_LEN)
DEV=windows_for_split('validation',DEV_BLOCKS,SEQ_LEN)
TEST=windows_for_split('test',TEST_BLOCKS,SEQ_LEN)
DATA_HASHES={name:hashlib.sha256(t.numpy().tobytes()).hexdigest() for name,t in
             [('train',TRAIN),('validation',DEV),('test',TEST)]}
assert len(set(DATA_HASHES.values()))==3
print('data',TRAIN.shape,DEV.shape,TEST.shape,'hashes',DATA_HASHES)

## 5 · Capture frozen native full-attention teacher activations
For each candidate target layer, record mixer inputs, teacher mixer output and the exact rotary embeddings. Native Qwen produces the common targets once; candidate optimization never updates the teacher.

In [ ]:
TEACHER={i:{} for i in TARGET_LAYERS}

@torch.inference_mode()
def forward_native(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(split,ids):
    rows={i:{'x':[],'y':[],'cos':[],'sin':[]} for i in TARGET_LAYERS}
    hooks=[]
    for i in TARGET_LAYERS:
        def fn(module,args,kwargs,out,idx=i):
            x=kwargs['hidden_states']
            y=out[0] if isinstance(out,tuple) else out
            cos,sin=kwargs['position_embeddings']
            rows[idx]['x'].append(x.detach().cpu().to(torch.float16))
            rows[idx]['y'].append(y.detach().cpu().to(torch.float16))
            rows[idx]['cos'].append(cos.detach().cpu().to(torch.float32))
            rows[idx]['sin'].append(sin.detach().cpu().to(torch.float32))
        hooks.append(LAYERS[i].self_attn.register_forward_hook(fn,with_kwargs=True))
    try:
        for j in range(0,len(ids),BATCH_SIZE):
            forward_native(ids[j:j+BATCH_SIZE],keep=1)
    finally:
        for h in hooks: h.remove()
    for i in TARGET_LAYERS:
        TEACHER[i][split]={key:torch.cat(val,dim=0).contiguous() for key,val in rows[i].items()}
        print(split,'layer',i,'x/y',TEACHER[i][split]['x'].shape,
              'teacher_y_rms',TEACHER[i][split]['y'].float().square().mean().sqrt().item())

capture('train',TRAIN)
capture('dev',DEV)
torch.cuda.empty_cache()

## 6 · Build V3.1 controls and V4 graph candidates
Both use the same tested Qwen3.5 attention adapter. Only the memory core changes. Trainable parameters are restricted to `adapter.core`; pretrained Qwen weights are frozen.

In [ ]:
def make_adapter(idx,name):
    native=LAYERS[idx].self_attn
    dims=dict(num_heads=native.q_proj.out_features//(2*native.head_dim),
              num_kv_heads=native.k_proj.out_features//native.head_dim,
              head_dim=native.head_dim,feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE)
    opt=CANDIDATES[name].copy();family=opt.pop('family')
    if family=='v3':
        core=CeNNDeltaCellV3(**dims,**opt)
    else:
        core=CeNNMemoryGraphV4(**dims,top_k=GRAPH_K,edge_dim=EDGE_DIM,**opt)
    core=core.to(DEVICE)
    adapter=Qwen35DeltaCellAttention(native,core)
    assert all(not p.requires_grad for p in native.parameters())
    return adapter,dict(**dims,**opt,family=family)

@contextmanager
def graft(idx,adapter):
    native=LAYERS[idx].self_attn
    assert native is adapter.original
    LAYERS[idx].self_attn=adapter
    try: yield
    finally: LAYERS[idx].self_attn=native

def batch_capture(idx,split,indices):
    d=TEACHER[idx][split]
    return (d['x'][indices].to(DEVICE,dtype=DTYPE),
            d['y'][indices].to(DEVICE,dtype=DTYPE),
            (d['cos'][indices].to(DEVICE),d['sin'][indices].to(DEVICE)))

def local_measures(pred,target):
    p,t=pred.float(),target.float()
    return ((p-t).square().mean()/t.square().mean().clamp_min(1e-8),
            F.cosine_similarity(p,t,dim=-1).mean())

@torch.no_grad()
def local_validation(adapter,idx):
    adapter.eval();err=energy=cs=0.;count=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        indices=torch.arange(begin,min(begin+BATCH_SIZE,DEV_BLOCKS))
        x,y,rope=batch_capture(idx,'dev',indices)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
        pp=pred.float(); yy=y.float()
        err+=(pp-yy).square().sum().item();energy+=yy.square().sum().item()
        cs+=F.cosine_similarity(pp,yy,dim=-1).sum().item()
        count+=pp.shape[0]*pp.shape[1]
    return dict(nmse=err/max(energy,1e-8),cosine=cs/count)

@torch.no_grad()
def snapshot_core(adapter):
    return {k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}

LOSS_COS_W=.15

## 7 · Fair two-stage local distillation
Stage A optimizes **one shared** Delta+Local32 foundation. Stage B initializes each candidate from those same weights, uses identical sampled training indices, and performs equal numbers of post-warmstart steps. This isolates *incremental* graph benefits. Checkpoints are selected on local **validation** NMSE/cosine, never on TEST performance.

A fixed seed makes this reproducible; publication claims require additional seeds and a held-out hyperparameter process.

In [ ]:
def fit(idx,name,steps,shared_state=None,seed=SEED,verbose=True):
    torch.manual_seed(seed+idx)   # identical candidate sampling/RNG budget
    adapter,config=make_adapter(idx,name)
    if shared_state is not None:
        missing,unexpected=adapter.core.load_state_dict(shared_state,strict=False)
        assert not unexpected,unexpected
        if name.startswith('graph_'):
            assert missing and all(k.startswith(('edge_','log_basis','log_content',
                                                'graph_strength','cell_rate','query_local_router'))
                                   or k=='shuffle_perm' for k in missing),missing
        else:
            assert not missing,missing
    optim=torch.optim.AdamW(adapter.core.parameters(),lr=LR,weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda',enabled=DTYPE==torch.float16)
    best=float('inf');best_sd=None;best_metrics=None
    for step in range(1,steps+1):
        indices=torch.randint(0,CALIB_BLOCKS,(BATCH_SIZE,))
        x,y,rope=batch_capture(idx,'train',indices)
        adapter.train();optim.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
            nmse,cos=local_measures(pred,y)
            loss=nmse+LOSS_COS_W*(1.-cos)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'Nonfinite loss at {idx}/{name}/{step}')
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(adapter.core.parameters(),1.)
        scaler.step(optim);scaler.update()
        if step==1 or step==steps or step%max(1,steps//5)==0:
            metrics=local_validation(adapter,idx)
            score=metrics['nmse']+LOSS_COS_W*(1.-metrics['cosine'])
            if score<best:
                best=score;best_sd=snapshot_core(adapter);best_metrics=metrics
            if verbose:
                print(f'L{idx} {name} {step}/{steps} loss={loss.item():.4f} '
                      f'cos={metrics["cosine"]:.4f} nmse={metrics["nmse"]:.4f}',flush=True)
    adapter.core.load_state_dict(best_sd);adapter.eval()
    return adapter,config,best_metrics,best_sd

SHARED={}
for idx in TARGET_LAYERS:
    shared,shared_cfg,metrics,weights=fit(idx,'delta_local32',SHARED_STEPS)
    SHARED[idx]=weights
    torch.save({'layer':idx,'core_state':weights,'config':shared_cfg,'steps':SHARED_STEPS,
                'source_git_sha':GIT_SHA},OUT/'checkpoints'/f'shared_delta_local32_L{idx}.pt')
    print('Shared base L',idx,metrics)
    del shared,weights;gc.collect();torch.cuda.empty_cache()

## 8 · Validation quality and controlled GPU prefill measurement
For better statistical power than the previous V3 quick screen, compute **every next-token position** (all positions 0…T-2) instead of every eighth token. Store only the teacher's top-1 token ID per position to avoid keeping hundreds of MB of full logits; consequently this notebook does **not** report full-distribution KL.

GPU timings use CUDA events, synchronization, warmups and median/IQR. Early candidate timing is exploratory; the held-out finalist receives **interleaved paired measurements** to control run-order effects.

In [ ]:
EVAL_POSITIONS=torch.arange(0,SEQ_LEN-1,device=DEVICE,dtype=torch.long)

@torch.inference_mode()
def evaluate_logits(ids,teacher_top1=None):
    n=0;loss_sum=0.;gold=0;agree=0;references=[]
    for start in range(0,len(ids),BATCH_SIZE):
        blocks=ids[start:start+BATCH_SIZE]
        positions=torch.arange(blocks.shape[1]-1,device=DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=blocks.to(DEVICE),use_cache=False,
                         logits_to_keep=positions).logits.float()
        truth=blocks[:,1:].to(DEVICE)
        assert logits.shape[:2]==truth.shape,(logits.shape,truth.shape)
        logp=F.log_softmax(logits,dim=-1)
        loss_sum-=logp.gather(-1,truth.unsqueeze(-1)).sum().item()
        predictions=logits.argmax(-1)
        gold+=(predictions==truth).sum().item();n+=truth.numel()
        if teacher_top1 is None:
            references.append(predictions.detach().cpu().to(torch.int32))
        else:
            true_predictions=teacher_top1[start//BATCH_SIZE].to(DEVICE)
            assert true_predictions.shape==predictions.shape
            agree+=(predictions==true_predictions).sum().item()
        del logits,logp
    metrics=dict(nll=loss_sum/n,next_token_accuracy=gold/n,decisions=n)
    if teacher_top1 is not None: metrics['teacher_agreement']=agree/n
    return metrics,references

@torch.inference_mode()
def timed_prefill(ids,repeats=9,warmups=4):
    sample=ids[:min(BATCH_SIZE,len(ids))].to(DEVICE)
    for _ in range(warmups):
        with torch.autocast('cuda',dtype=DTYPE):
            model(input_ids=sample,use_cache=False,logits_to_keep=1)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    measurements=[]
    for r in range(repeats):
        start=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.autocast('cuda',dtype=DTYPE):
            out=model(input_ids=sample,use_cache=False,logits_to_keep=1)
        end.record();torch.cuda.synchronize()
        assert bool(torch.isfinite(out.logits).all())
        measurements.append(start.elapsed_time(end))
    p25,p75=np.percentile(measurements,[25,75])
    return dict(median_latency_ms=float(np.median(measurements)),
                iqr_latency_ms=float(p75-p25),
                peak_gpu_mib=float(torch.cuda.max_memory_allocated()/1024**2),
                repeats=repeats)

NATIVE_DEV,NATIVE_DEV_TOP1=evaluate_logits(DEV)
NATIVE_BENCH=timed_prefill(DEV)
print('Native validation:',NATIVE_DEV,'Native prefill:',NATIVE_BENCH)

## 9 · Candidate comparison: same shared initialization and training budget
Run 5 architectures on each selected layer; save candidate checkpoints, validation metrics, parameter counts, timing and exact run metadata. All students are evaluated inside a temporary single-mixer graft and leave the original model unchanged afterward.

In [ ]:
REPORT=[]
for idx in TARGET_LAYERS:
    for name in CANDIDATES:
        print(f'\n========== layer {idx}, {name} ==========',flush=True)
        student,cfg,loc,weights=fit(idx,name,CANDIDATE_STEPS,shared_state=SHARED[idx])
        with graft(idx,student):
            dev,_=evaluate_logits(DEV,teacher_top1=NATIVE_DEV_TOP1)
            bench=timed_prefill(DEV,repeats=5,warmups=2)
        row=dict(layer=int(idx),variant=name,**cfg,
                 shared_steps=SHARED_STEPS,candidate_steps=CANDIDATE_STEPS,
                 trainable_core_params=sum(p.numel() for p in student.core.parameters()),
                 local_dev_nmse=loc['nmse'],local_dev_cosine=loc['cosine'],
                 dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
                 dev_agreement=dev['teacher_agreement'],val_decisions=dev['decisions'],
                 prefill_ms=bench['median_latency_ms'],prefill_iqr_ms=bench['iqr_latency_ms'],
                 prefill_speedup=NATIVE_BENCH['median_latency_ms']/bench['median_latency_ms'],
                 peak_gpu_mib=bench['peak_gpu_mib'])
        REPORT.append(row)
        torch.save({'core_state':weights,'core_config':cfg,'model_id':MODEL_ID,
                    'model_revision':REVISION,'source_git_sha':GIT_SHA,
                    'layer':idx,'variant':name,'no_cache':True},
                   OUT/'checkpoints'/f'v4_layer{idx}_{name}.pt')
        pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
        print(json.dumps(row,indent=2),flush=True)
        del student,weights;gc.collect();torch.cuda.empty_cache()
print('Candidates completed:',len(REPORT))

## 10 · Validation-only acceptance and winner selection
Preset gates: local cosine ≥0.95; ΔNLL ≤+0.10 nats/token; teacher top-1 agreement ≥0.98. The gate is intentionally more demanding than V3's 0.90 agreement. Candidates failing a gate are diagnostic only; **native Qwen remains the production model**. Test is not read here.

In [ ]:
MIN_COSINE=.95
MAX_DELTA_NLL=.10
MIN_AGREEMENT=.98
ranked=pd.DataFrame(REPORT)
ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                        (ranked.dev_delta_nll<=MAX_DELTA_NLL)&
                        (ranked.dev_agreement>=MIN_AGREEMENT))
ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                          ascending=[False,True,True]).reset_index(drop=True)
display(ranked[['layer','variant','local_dev_cosine','local_dev_nmse','dev_delta_nll',
                'dev_agreement','prefill_speedup','quality_gate']])
qualified=ranked[ranked.quality_gate]
if len(qualified):
    winner=qualified.sort_values(['prefill_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
    quality_accepted=True
else:
    winner=ranked.iloc[0]
    quality_accepted=False
print('VALIDATION-SELECTED DIAGNOSTIC:',winner.layer,winner.variant,
      'quality gate accepted?',quality_accepted)
ranked.to_csv(OUT/'validation_ranked.csv',index=False)

## 11 · Frozen-weights topology intervention (validation only)
For the **best trained learned-graph candidate**, evaluate the same adapter with only its graph neighbor indices permuted; DO NOT retrain. This provides a more meaningful graph-topology control than a separately trained shuffled graph. If fidelity barely changes, there is little evidence that learned connectivity matters.

In [ ]:
GRAPH_INTERVENTION=None
learned=ranked[ranked.variant.isin(['graph_static_local32','graph_adaptive_local32'])]
if len(learned):
    graph_pick=learned.iloc[0]
    gp=torch.load(OUT/'checkpoints'/f'v4_layer{int(graph_pick.layer)}_{graph_pick.variant}.pt',
                  map_location='cpu',weights_only=True)
    graph_student,_=make_adapter(int(graph_pick.layer),str(graph_pick.variant))
    graph_student.core.load_state_dict(gp['core_state']);graph_student.eval()
    def graph_dev_metrics(mode):
        graph_student.core.graph_mode=mode
        local=local_validation(graph_student,int(graph_pick.layer))
        with graft(int(graph_pick.layer),graph_student):
            report,_=evaluate_logits(DEV,teacher_top1=NATIVE_DEV_TOP1)
        return dict(local=local,whole_model=report)
    normal=graph_dev_metrics('learned')
    shuffled=graph_dev_metrics('shuffled')
    graph_student.core.graph_mode='learned'
    GRAPH_INTERVENTION=dict(layer=int(graph_pick.layer),variant=str(graph_pick.variant),
                            trained_learned=normal,frozen_shuffled=shuffled,
                            cosine_loss=normal['local']['cosine']-shuffled['local']['cosine'],
                            delta_nll=shuffled['whole_model']['nll']-normal['whole_model']['nll'])
    print(json.dumps(GRAPH_INTERVENTION,indent=2))
    del graph_student;torch.cuda.empty_cache()

## 12 · One held-out test audit + **paired alternating** latency comparison
Select strictly from validation first. Then load exactly one adapter for the held-out test. Paired latency alternates **native→student** and **student→native** to reduce order bias, with the same GPU, batch, sequence length and autocast settings. The result passes only if quality gates survive **and** the paired median speedup exceeds 1.05×.

In [ ]:
WIN_LAYER=int(winner.layer)
WIN_NAME=str(winner.variant)
payload=torch.load(OUT/'checkpoints'/f'v4_layer{WIN_LAYER}_{WIN_NAME}.pt',
                   map_location='cpu',weights_only=True)
selected,_=make_adapter(WIN_LAYER,WIN_NAME)
selected.core.load_state_dict(payload['core_state']);selected.eval()

@torch.inference_mode()
def paired_timing(idx,student,blocks,repeats=14,warmups=3):
    sample=blocks[:min(BATCH_SIZE,len(blocks))].to(DEVICE)
    native_times=[];student_times=[]
    def run_one(use_student):
        with (graft(idx,student) if use_student else nullcontext()):
            start=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            start.record()
            with torch.autocast('cuda',dtype=DTYPE):
                out=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(out).all())
            return start.elapsed_time(end)
    for i in range(warmups+repeats):
        if i%2==0:
            a=run_one(False);b=run_one(True)
        else:
            b=run_one(True);a=run_one(False)
        if i>=warmups:
            native_times.append(a);student_times.append(b)
    return dict(native_ms=float(np.median(native_times)),
                student_ms=float(np.median(student_times)),
                speedup=float(np.median(native_times)/np.median(student_times)),
                native_iqr_ms=float(np.subtract(*np.percentile(native_times,[75,25]))),
                student_iqr_ms=float(np.subtract(*np.percentile(student_times,[75,25]))),
                repeats=repeats)

NATIVE_TEST,NATIVE_TEST_TOP1=evaluate_logits(TEST)
with graft(WIN_LAYER,selected):
    STUDENT_TEST,_=evaluate_logits(TEST,teacher_top1=NATIVE_TEST_TOP1)
TEST_DELTA_NLL=STUDENT_TEST['nll']-NATIVE_TEST['nll']
PAIRED_TEST=paired_timing(WIN_LAYER,selected,TEST)
FINAL_PASS=bool(quality_accepted and TEST_DELTA_NLL<=MAX_DELTA_NLL and
                STUDENT_TEST['teacher_agreement']>=MIN_AGREEMENT and
                PAIRED_TEST['speedup']>1.05)
AUDIT=dict(model=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,
           architecture='CeNN-MemoryGraph V4',profile=PROFILE,
           selected=dict(layer=WIN_LAYER,variant=WIN_NAME),
           seed=SEED,seq_len=SEQ_LEN,train_steps=dict(shared=SHARED_STEPS,candidate=CANDIDATE_STEPS),
           quality_gates=dict(min_cosine=MIN_COSINE,max_delta_nll=MAX_DELTA_NLL,
                              min_agreement=MIN_AGREEMENT,min_speedup=1.05),
           validation_accepted=quality_accepted,final_pass=FINAL_PASS,
           validation_candidates=REPORT,graph_topology_intervention=GRAPH_INTERVENTION,
           test_native=NATIVE_TEST,test_student=STUDENT_TEST,
           test_delta_nll=TEST_DELTA_NLL,paired_test_timing=PAIRED_TEST,
           data_fingerprints=DATA_HASHES,
           constraints='Qwen full-attention text only; cache-free; unpadded; reference eager GPU. '
                       'Not an autoregressive decode benchmark, no fused Triton kernel. '
                       'Single seed and WikiText2 only; no evidence of publication-level novelty.')
(OUT/'final_audit.json').write_text(json.dumps(AUDIT,indent=2,allow_nan=False))
print(json.dumps({key:AUDIT[key] for key in
       ['selected','validation_accepted','final_pass','test_native',
        'test_student','test_delta_nll','paired_test_timing']},indent=2))
if not FINAL_PASS:
    print('REJECT for deployment: retain original Qwen layer 11.')

## 13 · Optional prefill scaling by sequence length
Benchmark the selected diagnostic candidate at 128, 256, 512, and 1024 tokens **without using held-out test targets**. This measures pure cache-free prefill, not token-by-token generation. The per-length benchmarks are exploratory and should be repeated with a fused kernel and >=3 independent runs before speed claims.

In [ ]:
# @title Optional length sweep
RUN_LENGTH_SWEEP = False # @param {type:'boolean'}
LENGTH_RESULTS=[]
if RUN_LENGTH_SWEEP:
    for length in [128,256,512,1024]:
        windows=windows_for_split('validation',4,length)
        sample=paired_timing(WIN_LAYER,selected,windows,repeats=8,warmups=2)
        LENGTH_RESULTS.append(dict(seq_len=length,**sample))
        print(length, sample)
    display(pd.DataFrame(LENGTH_RESULTS))
    pd.DataFrame(LENGTH_RESULTS).to_csv(OUT/'prefill_length_sweep.csv',index=False)
    AUDIT['length_sweep']=LENGTH_RESULTS
    (OUT/'final_audit.json').write_text(json.dumps(AUDIT,indent=2,allow_nan=False))

## 14 · Plot, export and record reproducible result package
Keep each checkpoint as an **adapter-only** experimental state dict, referencing the exact base Qwen model SHA. The native model was never modified on disk. Download the ZIP and share `final_audit.json`, `validation_candidates.csv`, and `validation_ranked.csv` for next-step analysis.

In [ ]:
import shutil
frame=pd.DataFrame(REPORT)
fig,ax=plt.subplots(figsize=(11,4))
for label,group in frame.groupby('variant'):
    ax.scatter(group.local_dev_cosine,group.dev_delta_nll,s=75,label=label)
ax.axvline(MIN_COSINE,linestyle='--',color='gray')
ax.axhline(MAX_DELTA_NLL,linestyle='--',color='gray')
ax.set(xlabel='Local teacher cosine (higher better)',ylabel='Validation ΔNLL (lower better)',
       title='CeNN-MemoryGraph V4: semantic graph vs Delta+Local32')
ax.legend(fontsize=8);fig.tight_layout()
fig.savefig(OUT/'quality_ablation.png',dpi=160)
(OUT/'manifest.json').write_text(json.dumps({
  'source_git_sha':GIT_SHA,'base_model':MODEL_ID,'base_model_revision':REVISION,
  'notebook':'Qwen35_08B_CeNN_MemoryGraph_V4_Colab.ipynb',
  'experiment_architecture':'CeNN-MemoryGraph V4','profile':PROFILE,'candidate_names':list(CANDIDATES),
  'use_cache':False,'held_out_split':'WikiText2 test','reusable_python_module':
  'tinycenn_lm.cenn_memorygraph_v4'},indent=2))
shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Audit:',OUT/'final_audit.json')
print('Archive:',str(OUT)+'.zip')
from IPython.display import display,FileLink
display(FileLink(str(OUT)+'.zip'))

### How to interpret the results

1. **First compare** `delta_local32` to `v3_adaptive_local32`: did V3 cellular diffusion help once Local32 is controlled?
2. **Then compare** `graph_static_local32` to `delta_local32`: does a learned semantic graph improve fidelity?
3. **Then compare** `graph_adaptive_local32` to `graph_static_local32`: do content weighting + per-token routing add measurable value?
4. **Inspect the intervention**: the same trained graph has its neighbors shuffled *without retraining*. A near-zero effect undermines evidence for learned topology.
5. **Finally inspect paired test timing**: a single reference PyTorch graph is not a fused kernel. Native performance is the benchmark, not a promise of acceleration.

This notebook's fixed top-k index selection has a discrete edge-choice operation: selected edge *weights* have gradients, but gradients do not pass through the combinatorial top-k choices. Publication-worthy novelty requires comparisons to routed-memory and sparse-state prior art, longer retrieval tasks, multiple seeds, calibrated parameter budgets and fused GPU kernels.

**Do not describe a rejected candidate as a faster model.** Keep Qwen native unless the predeclared quality and speed gates both pass.